## Introduction

TBD

 ## Python Imports

In [221]:
# Standard Library
import os
import sys
import warnings

from pathlib import Path

# Data Handling
import pandas as pd

# Suppress warnings
warnings.filterwarnings("ignore")

In [222]:
# Add the source subdirectory to the system path to allow import config from settings.py
current_directory = Path(os.getcwd())
website_base_directory = current_directory.parent.parent.parent
src_directory = website_base_directory / "src"
sys.path.append(str(src_directory)) if str(src_directory) not in sys.path else None

# Import settings.py
from settings import config

# Add configured directories from config to path
SOURCE_DIR = config("SOURCE_DIR")
sys.path.append(str(Path(SOURCE_DIR))) if str(Path(SOURCE_DIR)) not in sys.path else None

# Add other configured directories
BASE_DIR = config("BASE_DIR")
CONTENT_DIR = config("CONTENT_DIR")
POSTS_DIR = config("POSTS_DIR")
PAGES_DIR = config("PAGES_DIR")
PUBLIC_DIR = config("PUBLIC_DIR")
SOURCE_DIR = config("SOURCE_DIR")
DATA_DIR = config("DATA_DIR")
DATA_MANUAL_DIR = config("DATA_MANUAL_DIR")

## Python Functions

Here are the functions needed for this project:

* [load_data](/posts/reusable-extensible-python-functions-financial-data-analysis/#load_data): Load data from a CSV, Excel, or Pickle file into a pandas DataFrame.

In [223]:
from load_data import load_data

## Set Variables

In [224]:
ticker = "XLY"
source = "Polygon"
asset_class = "Exchange_Traded_Funds"
timeframe = "day"

## Load Post-Split Data

In [225]:
df_post_split = load_data(
    base_directory=DATA_DIR,
    ticker=f"{ticker}-postsplit",
    source=source,
    asset_class=asset_class,
    timeframe=timeframe,
    file_format="pickle",
)

df_post_split

,Date,open,high,low,close,volume,vwap,transactions,otc
0,2024-04-04 04:00:00,90.660,90.9975,88.9125,88.925,9.650556e+06,89.7407,68143,None
1,2024-04-05 04:00:00,89.220,90.0000,89.0350,89.420,1.008189e+07,89.4918,74277,None
2,2024-04-08 04:00:00,89.950,90.5650,89.8800,90.270,8.295708e+06,90.3008,56303,None
3,2024-04-09 04:00:00,90.715,90.7450,89.9400,90.695,8.562278e+06,90.3510,59768,None
4,2024-04-10 04:00:00,89.220,89.6600,88.8800,89.295,1.080596e+07,89.2723,83876,None
...,...,...,...,...,...,...,...,...,...
496,2026-03-27 04:00:00,108.310,108.3100,105.4450,105.680,1.045473e+07,106.3768,105746,None
497,2026-03-30 04:00:00,106.610,107.1650,105.1900,105.660,1.019198e+07,106.1202,106059,None
498,2026-03-31 04:00:00,107.230,109.4100,106.7100,108.980,1.948755e+07,108.5157,150928,None
499,2026-04-01 04:00:00,109.690,110.5000,108.7350,109.800,1.446761e+07,109.7929,112810,None


## Calc Cutoff and Split Check Dates

In [226]:
cutoff_date = df_post_split.loc[3, "Date"]
print(f"Cutoff date for split adjustment: {cutoff_date}")
split_check_date = df_post_split.loc[4, "Date"]
print(f"Split check date: {split_check_date}")

Cutoff date for split adjustment: 2024-04-09 04:00:00
Split check date: 2024-04-10 04:00:00


## Load Pre-Split Data

In [227]:
df_pre_split = load_data(
    base_directory=DATA_DIR,
    ticker=f"{ticker}-presplit",
    source=source,
    asset_class=asset_class,
    timeframe=timeframe,
    file_format="pickle",
)

print(df_pre_split.shape)
df_pre_split[df_pre_split["Date"] < cutoff_date]

(558, 9)


,Date,open,high,low,close,volume,vwap,transactions,otc
0,2023-08-28 04:00:00,165.72,165.890,164.330,165.32,2981442.0,165.0886,37438,None
1,2023-08-29 04:00:00,165.28,169.520,165.280,169.45,4457096.0,168.2195,48208,None
2,2023-08-30 04:00:00,169.14,170.820,168.720,170.18,3644587.0,169.8853,46701,None
3,2023-08-31 04:00:00,170.29,171.640,170.050,170.71,3936448.0,170.9943,41870,None
4,2023-09-01 04:00:00,171.68,172.120,168.680,169.67,5309363.0,169.8699,58785,None
...,...,...,...,...,...,...,...,...,...
149,2024-04-02 04:00:00,179.58,179.920,179.020,179.83,6135372.0,179.5877,68841,None
150,2024-04-03 04:00:00,178.82,180.590,178.820,179.96,5082613.0,179.9669,57062,None
151,2024-04-04 04:00:00,181.32,181.995,177.825,177.85,4825278.0,179.4813,68143,None
152,2024-04-05 04:00:00,178.44,180.000,178.070,178.84,5040947.0,178.9835,74277,None


## Calc Split Ratio

In [228]:
# Extract "open" price from the same row from both pre and post split dataframes to calculate the split ratio
pre_split_open_price = df_pre_split[df_pre_split["Date"] == split_check_date]["open"].values[0]
post_split_open_price = df_post_split[df_post_split["Date"] == split_check_date]["open"].values[0]
SPLIT_RATIO = pre_split_open_price / post_split_open_price
print(f"Calculated split ratio: {SPLIT_RATIO}")

Calculated split ratio: 2.0


## Calc Corrected Pre-Split Data

In [229]:
df_pre_split_corrected = df_pre_split.copy()
df_pre_split_corrected["open"] = df_pre_split_corrected["open"] / SPLIT_RATIO
df_pre_split_corrected["high"] = df_pre_split_corrected["high"] / SPLIT_RATIO
df_pre_split_corrected["low"] = df_pre_split_corrected["low"] / SPLIT_RATIO
df_pre_split_corrected["close"] = df_pre_split_corrected["close"] / SPLIT_RATIO
df_pre_split_corrected["volume"] = df_pre_split_corrected["volume"] * SPLIT_RATIO
df_pre_split_corrected["vwap"] = df_pre_split_corrected["vwap"] / SPLIT_RATIO
df_pre_split_corrected[df_pre_split_corrected["Date"] < cutoff_date]

,Date,open,high,low,close,volume,vwap,transactions,otc
0,2023-08-28 04:00:00,82.860,82.9450,82.1650,82.660,5962884.0,82.54430,37438,None
1,2023-08-29 04:00:00,82.640,84.7600,82.6400,84.725,8914192.0,84.10975,48208,None
2,2023-08-30 04:00:00,84.570,85.4100,84.3600,85.090,7289174.0,84.94265,46701,None
3,2023-08-31 04:00:00,85.145,85.8200,85.0250,85.355,7872896.0,85.49715,41870,None
4,2023-09-01 04:00:00,85.840,86.0600,84.3400,84.835,10618726.0,84.93495,58785,None
...,...,...,...,...,...,...,...,...,...
149,2024-04-02 04:00:00,89.790,89.9600,89.5100,89.915,12270744.0,89.79385,68841,None
150,2024-04-03 04:00:00,89.410,90.2950,89.4100,89.980,10165226.0,89.98345,57062,None
151,2024-04-04 04:00:00,90.660,90.9975,88.9125,88.925,9650556.0,89.74065,68143,None
152,2024-04-05 04:00:00,89.220,90.0000,89.0350,89.420,10081894.0,89.49175,74277,None


## Combine Pre- and Post-Split Data

In [230]:
display_date = df_post_split.loc[5, "Date"]
combined = pd.concat([df_pre_split_corrected[df_pre_split_corrected["Date"] <= cutoff_date], df_post_split], ignore_index=True)
combined[combined["Date"] <= display_date].tail(15)

,Date,open,high,low,close,volume,vwap,transactions,otc
146,2024-03-27 04:00:00,91.885,92.2400,91.4350,92.230,5811948.0,91.83250,44414,None
147,2024-03-28 04:00:00,92.115,92.3850,91.9150,91.945,5958102.0,92.12120,48606,None
148,2024-04-01 04:00:00,92.045,92.1450,90.9250,91.260,9941710.0,91.25955,57190,None
149,2024-04-02 04:00:00,89.790,89.9600,89.5100,89.915,12270744.0,89.79385,68841,None
150,2024-04-03 04:00:00,89.410,90.2950,89.4100,89.980,10165226.0,89.98345,57062,None
151,2024-04-04 04:00:00,90.660,90.9975,88.9125,88.925,9650556.0,89.74065,68143,None
152,2024-04-05 04:00:00,89.220,90.0000,89.0350,89.420,10081894.0,89.49175,74277,None
153,2024-04-08 04:00:00,89.950,90.5650,89.8800,90.270,8295708.0,90.30080,56303,None
154,2024-04-09 04:00:00,90.715,90.7450,89.9400,90.695,8562278.0,90.35095,59768,None
155,2024-04-04 04:00:00,90.660,90.9975,88.9125,88.925,9650556.0,89.74070,68143,None


## Confirm Rows To Be Dropped

In [231]:
print(f"Should drop the following duplicate rows if they exist:")
combined[combined.round(3).duplicated(subset=["Date", "open", "high", "low", "close", "volume"], keep=False)]

Should drop the following duplicate rows if they exist:


,Date,open,high,low,close,volume,vwap,transactions,otc
151,2024-04-04 04:00:00,90.660,90.9975,88.9125,88.925,9650556.0,89.74065,68143,None
152,2024-04-05 04:00:00,89.220,90.0000,89.0350,89.420,10081894.0,89.49175,74277,None
153,2024-04-08 04:00:00,89.950,90.5650,89.8800,90.270,8295708.0,90.30080,56303,None
154,2024-04-09 04:00:00,90.715,90.7450,89.9400,90.695,8562278.0,90.35095,59768,None
155,2024-04-04 04:00:00,90.660,90.9975,88.9125,88.925,9650556.0,89.74070,68143,None
156,2024-04-05 04:00:00,89.220,90.0000,89.0350,89.420,10081894.0,89.49180,74277,None
157,2024-04-08 04:00:00,89.950,90.5650,89.8800,90.270,8295708.0,90.30080,56303,None
158,2024-04-09 04:00:00,90.715,90.7450,89.9400,90.695,8562278.0,90.35100,59768,None


## Drop Duplicate Rows

In [232]:
combined = pd.concat([df_pre_split_corrected[df_pre_split_corrected["Date"] <= cutoff_date], df_post_split], ignore_index=True).round(3).drop_duplicates(subset=["Date", "open", "high", "low", "close", "volume"], keep="last")
combined[combined["Date"] <= display_date].tail(15)

,Date,open,high,low,close,volume,vwap,transactions,otc
142,2024-03-21 04:00:00,92.270,92.610,92.120,92.165,7146376.0,92.331,48125,None
143,2024-03-22 04:00:00,91.205,91.602,91.015,91.360,5719002.0,91.311,43959,None
144,2024-03-25 04:00:00,91.005,91.530,91.005,91.060,5664684.0,91.209,41151,None
145,2024-03-26 04:00:00,91.645,92.025,91.112,91.175,5387844.0,91.608,40170,None
146,2024-03-27 04:00:00,91.885,92.240,91.435,92.230,5811948.0,91.832,44414,None
147,2024-03-28 04:00:00,92.115,92.385,91.915,91.945,5958102.0,92.121,48606,None
148,2024-04-01 04:00:00,92.045,92.145,90.925,91.260,9941710.0,91.260,57190,None
149,2024-04-02 04:00:00,89.790,89.960,89.510,89.915,12270744.0,89.794,68841,None
150,2024-04-03 04:00:00,89.410,90.295,89.410,89.980,10165226.0,89.983,57062,None
155,2024-04-04 04:00:00,90.660,90.998,88.912,88.925,9650556.0,89.741,68143,None


## Reset Index And Display Data

In [233]:
combined.reset_index(drop=True, inplace=True)
combined

,Date,open,high,low,close,volume,vwap,transactions,otc
0,2023-08-28 04:00:00,82.860,82.945,82.165,82.660,5.962884e+06,82.544,37438,None
1,2023-08-29 04:00:00,82.640,84.760,82.640,84.725,8.914192e+06,84.110,48208,None
2,2023-08-30 04:00:00,84.570,85.410,84.360,85.090,7.289174e+06,84.943,46701,None
3,2023-08-31 04:00:00,85.145,85.820,85.025,85.355,7.872896e+06,85.497,41870,None
4,2023-09-01 04:00:00,85.840,86.060,84.340,84.835,1.061873e+07,84.935,58785,None
...,...,...,...,...,...,...,...,...,...
647,2026-03-27 04:00:00,108.310,108.310,105.445,105.680,1.045473e+07,106.377,105746,None
648,2026-03-30 04:00:00,106.610,107.165,105.190,105.660,1.019198e+07,106.120,106059,None
649,2026-03-31 04:00:00,107.230,109.410,106.710,108.980,1.948755e+07,108.516,150928,None
650,2026-04-01 04:00:00,109.690,110.500,108.735,109.800,1.446761e+07,109.793,112810,None


## Export

In [234]:
# Export the combined DataFrame to a pickle file
combined.to_pickle(DATA_DIR / source / asset_class / timeframe / f"{ticker}.pkl")